# Make the duel music with AI

Makes the music for the scroll site with **MusicGen** (Meta's free music model): 30-second instrumental tracks, each shaped so it loops without a jump.

This chapter is an **Xyrus Academy** duel, so there are two moods:
* **academy**: the calm build-up before the fight (panels 1–5);
* **fight**: from "BURN" (panel 6) to the end.

Each mood comes in two styles: **orchestral** (cinematic battle soundtrack) and **dark** (the sound of the fans' reading playlist:
dark electronic and trap with heavy bass, and haunting music-box piano). That makes 2 academy and 4 fight tracks.
Listen at the bottom, pick one of each, and send them back (or the whole zip).

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Menu **Runtime → Run all**. It takes about 12 minutes, most of it the first download of the model.
3. A zip called `duel_music.zip` downloads at the end.

To try different music, change the descriptions in Step 2 and run Steps 2 to 4 again.
Every description says *instrumental, no vocals*: lyrics distract from reading.
MusicGen's model is free for non-commercial use, which fits a personal project.

### Step 1: load the music model (about 5 minutes the first time)

In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

MODEL = "facebook/musicgen-large"   # best quality; "facebook/musicgen-medium" is faster and lighter
gpu = torch.cuda.is_available()
processor = AutoProcessor.from_pretrained(MODEL)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float16 if gpu else torch.float32)
model = model.to("cuda" if gpu else "cpu").eval()
RATE = model.config.audio_encoder.sampling_rate
print("Loaded on the", "graphics card" if gpu else "processor (very slow: choose the T4 GPU runtime)")

### Step 2: describe the music

In [ ]:
# One track per description. Name: what the music should sound like.
# The name's first word says where it goes on the site: academy_... (panels 1-5) or fight_... (panel 6 on).
# The *_dark ones follow the fans' reading playlist (dark electronic / trap alt-pop, music-box piano);
# the others follow cinematic orchestral battle soundtracks.
TRACKS = {
    "academy_strings": "warm light fantasy academy theme, gentle strings and soft piano, hopeful and curious, "
                       "studio ghibli inspired, slow build of quiet tension, 90 bpm, instrumental, no vocals",
    "academy_dark": "haunting music box melody over dark piano and soft strings, slow trap beat with deep sub bass, "
                    "melancholic and mysterious, cinematic dark pop, 80 bpm, instrumental, no vocals",
    "fight_orchestral": "epic orchestral anime battle music, fast taiko drums and pounding toms, aggressive staccato strings, "
                        "brass stabs, heroic and intense sword duel, 150 bpm, instrumental, no vocals",
    "fight_choir": "dark epic battle theme, heavy war drums, wordless low choir, driving cello ostinato, "
                   "soaring horn melody, desperate high stakes, 140 bpm, instrumental",
    "fight_dark": "dark cinematic electronic, heavy distorted 808 bass, hard trap drums, gritty synths, "
                  "aggressive build up and drop, villain energy, 140 bpm, instrumental, no vocals",
    "fight_dark_epic": "epic melodic dubstep, huge cinematic synth leads, dark atmosphere, punchy drums, "
                       "rising tension into a powerful drop, intense fight scene, 150 bpm, instrumental, no vocals",
}
SECONDS = 30   # MusicGen makes up to about 30 seconds at a time

### Step 3: make the tracks (about a minute each)

In [ ]:
import numpy as np, soundfile as sf, subprocess, os
from IPython.display import Audio, display

os.makedirs("duel_music", exist_ok=True)
for n, (name, description) in enumerate(TRACKS.items()):
    torch.manual_seed(n)
    inputs = processor(text=[description], padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        audio = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=int(SECONDS * 50))
    x = audio[0, 0].float().cpu().numpy()

    # make it loop: blend the last 1.5 seconds into the start, so the end runs straight back into the beginning
    fade = int(1.5 * RATE)
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    loop = x[:-fade].copy()
    loop[:fade] = x[:fade] * ramp + x[-fade:] * (1 - ramp)

    wav = f"duel_music/{name}.wav"
    sf.write(wav, loop, RATE)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-af", "loudnorm=I=-18:TP=-1.5",
                    "-ar", "44100", "-c:a", "libmp3lame", "-b:a", "128k", wav[:-4] + ".mp3"], check=True)
    os.remove(wav)
    print(f"{name}: {len(loop) / RATE:.0f} s")
    display(Audio(wav[:-4] + ".mp3"))

### Step 4: download

In [ ]:
from google.colab import files
!cd duel_music && zip -qr ../duel_music.zip .
files.download("duel_music.zip")